In [13]:
import os
import sys

In [15]:
dir_base = "../../"
print(os.path.abspath(dir_base))

c:\Users\grabn\Documents\CGV_2024\xeegVis_alll\all-in-on-eeg


In [16]:
sys.path.append(os.path.abspath(dir_base))

sys.path.append(os.path.abspath(os.path.join(dir_base, "backend", "experiments_xeegnet")))
sys.path.append(os.path.abspath(os.path.join(dir_base, "backend", "experiments_xeegnet", "shallownetXAI_main")))
sys.path.append(os.path.abspath(os.path.join(dir_base, "backend", "ml")))

In [18]:
%load_ext autoreload
%autoreload 2
from data_loading_expirements.load_data_pythonfun import *
from backend.ml.data_utils.load_data import *
from backend.ml.data_utils.prepare_data import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
dir_data = os.path.join(dir_base, "data", "datasets", "ds004504")
dir_data_abs = os.path.abspath(dir_data)
data_path = os.path.join(dir_data, "sub-001", "eeg", "sub-001_task-eyesclosed_eeg.set")

dir_shallownetXAI = os.path.join(dir_base, "backend", "experiments_xeegnet", "shallownetXAI_main")

In [24]:
disease_mapping = {"A": "Alzheimer Disease Group", "F": "Frontotemporal Dementia Group", "C": "Healthy Group"}

disease_encoding = {
    "A": 1,
    "F": 2,
    "C": 0,
}


df_metadata = load_metadata(dir_data=dir_data)

# metadata_path = os.path.join(dir_data, 'participants.tsv')
# df_metadata = pd.read_csv(metadata_path, sep='\t')
# df_metadata['group_long'] = df_metadata['Group'].map(disease_mapping)
# df_metadata['group_encoded'] = df_metadata['Group'].map(disease_encoding)
pd.set_option("display.max_rows", 4)
pd.set_option("display.max_columns", 20)
df_metadata

,participant_id,Gender,Age,Group,MMSE,group_long,group_encoded,participant_id_int
0,sub-001,F,57,A,16,Alzheimer Disease Group,1,1
1,sub-002,F,78,A,22,Alzheimer Disease Group,1,2
...,...,...,...,...,...,...,...,...
86,sub-087,M,73,F,24,Frontotemporal Dementia Group,2,87
87,sub-088,M,55,F,24,Frontotemporal Dementia Group,2,88


In [ ]:
print(f" number of FD participants: {len(participants_fd)}")
print(f" number of AD participants: {len(participants_ad)}")
print(f" number of healthy participants: {len(participants_healthy)}")

 number of FD participants: 23
 number of AD participants: 36
 number of HC participants: 29


In [ ]:
participants_healthy = df_metadata[df_metadata["Group"] == "C"]["participant_id_int"].tolist()
participants_fd = df_metadata[df_metadata["Group"] == "F"]["participant_id_int"].tolist()
participants_ad = df_metadata[df_metadata["Group"] == "A"]["participant_id_int"].tolist()

###
test_split = 0.2
n_participants = df_metadata["participant_id_int"].max()

print(int(n_participants * test_split))

n_test_participants = 18  # ~20.5% of 88, round up so divisible by 3 classes
n_test_participants_per_class = 6  # 3 classes: 6*3 = 18 test participants

n_splits = int(n_participants / n_test_participants)

print(f"n_participants: {n_participants}")
print(f"n_test_participants: {n_test_participants}")
print(f"n_splits: {n_splits}")

test_participants = {}

for i in range(n_splits):
    pass

17
n_participants: 88
n_test_participants: 18
n_splits: 4


In [35]:
n_participants / n_test_participants

np.float64(4.888888888888889)

In [43]:
import random

# ---- inputs (from your code) ----
participants_healthy = df_metadata[df_metadata["Group"] == "C"]["participant_id_int"].tolist()
participants_fd = df_metadata[df_metadata["Group"] == "F"]["participant_id_int"].tolist()
participants_ad = df_metadata[df_metadata["Group"] == "A"]["participant_id_int"].tolist()

n_test_per_class = 6
n_splits = 5
seed = 42

rng = random.Random(seed)


def make_class_splits(participants, n_test, n_splits, rng):
    """
    Returns a list of `n_splits` test-lists for one class.
    Every participant appears in a test set at least once (sampling without
    replacement). If participants run out before the last splits are filled,
    the leftover slots are filled by re-sampling from participants who have
    NOT yet appeared in the current refill cycle.
    """
    pool = participants[:]
    rng.shuffle(pool)

    splits = []
    idx = 0
    for s in range(n_splits):
        # take next n_test from the shuffled pool
        if idx + n_test <= len(pool):
            splits.append(pool[idx : idx + n_test])
            idx += n_test
        else:
            # not enough left: take what's left, then top up from a reshuffled
            # copy of participants who are NOT in this split yet
            taken = pool[idx:]
            needed = n_test - len(taken)
            remaining_candidates = [p for p in participants if p not in taken]
            rng.shuffle(remaining_candidates)
            taken += remaining_candidates[:needed]
            splits.append(taken)
            idx = len(pool)  # exhausted
    return splits


healthy_splits = make_class_splits(participants_healthy, n_test_per_class, n_splits, rng)
fd_splits = make_class_splits(participants_fd, n_test_per_class, n_splits, rng)
ad_splits = make_class_splits(participants_ad, n_test_per_class, n_splits, rng)

all_participants = set(participants_healthy + participants_fd + participants_ad)

test_participants = {}
val_participants = {}
train_participants = {}
for i in range(n_splits):
    test_set = healthy_splits[i] + fd_splits[i] + ad_splits[i]
    val_set = healthy_splits[(i - 1) % n_splits] + fd_splits[(i - 1) % n_splits] + ad_splits[(i - 1) % n_splits]

    test_participants[i] = test_set
    val_participants[i] = val_set
    train_participants[i] = sorted(all_participants - set(test_set) - set(val_set))

# ---- sanity checks ----
print(f"Total participants: {len(all_participants)}")
for i in range(n_splits):
    t = test_participants[i]
    v = val_participants[i]
    overlap_tv = set(t) & set(v)
    print(
        f"split {i}: "
        f"test={len(t)} (C:{sum(p in participants_healthy for p in t)}, "
        f"F:{sum(p in participants_fd for p in t)}, "
        f"A:{sum(p in participants_ad for p in t)}) | "
        f"val={len(v)}  (C:{sum(p in participants_healthy for p in v)}, "
        f"F:{sum(p in participants_fd for p in v)}, "
        f"A:{sum(p in participants_ad for p in v)}) | "
        f"train={len(train_participants[i])} | "
        f"test∩val={len(overlap_tv)}"
    )

# coverage: who never appears in any test set?
seen = set().union(*[set(test_participants[i]) for i in range(n_splits)])
missing = all_participants - seen
print(f"participants never in any test set: {len(missing)} -> {sorted(missing)}")

Total participants: 88
split 0: test=18 (C:6, F:6, A:6) | val=18  (C:6, F:6, A:6) | train=55 | test∩val=3
split 1: test=18 (C:6, F:6, A:6) | val=18  (C:6, F:6, A:6) | train=52 | test∩val=0
split 2: test=18 (C:6, F:6, A:6) | val=18  (C:6, F:6, A:6) | train=52 | test∩val=0
split 3: test=18 (C:6, F:6, A:6) | val=18  (C:6, F:6, A:6) | train=52 | test∩val=0
split 4: test=18 (C:6, F:6, A:6) | val=18  (C:6, F:6, A:6) | train=54 | test∩val=2
participants never in any test set: 6 -> [9, 15, 17, 24, 26, 28]


In [45]:
val_participants

{0: [45, 60, 37, 40, 57, 49, 66, 85, 84, 70, 83, 82, 5, 4, 2, 25, 3, 16],
 1: [55, 51, 47, 59, 58, 43, 66, 84, 71, 74, 73, 85, 35, 8, 10, 12, 19, 23],
 2: [42, 49, 48, 52, 46, 61, 82, 86, 80, 79, 81, 70, 7, 34, 31, 27, 11, 1],
 3: [56, 65, 53, 63, 38, 50, 88, 69, 76, 67, 87, 72, 30, 36, 32, 33, 18, 13],
 4: [39, 54, 64, 41, 62, 44, 77, 75, 83, 68, 78, 66, 29, 20, 14, 22, 6, 21]}

In [48]:
from pathlib import Path
import json

In [49]:
# ---- save splits to JSON ----
out_path = Path("data_splits.json")

splits_out = {
    str(i): {
        "train": sorted(int(p) for p in train_participants[i]),
        "val": sorted(int(p) for p in val_participants[i]),
        "test": sorted(int(p) for p in test_participants[i]),
    }
    for i in range(n_splits)
}

with out_path.open("w") as f:
    json.dump(splits_out, f, indent=2)

print(f"saved {n_splits} splits to {out_path.resolve()}")

saved 5 splits to C:\Users\grabn\Documents\CGV_2024\xeegVis_alll\all-in-on-eeg\backend\experiments_xeegnet\data_splits.json
